# Design Spaces 

## Setup

### Import Packages and Directories

In [ ]:
from pathlib import Path
import flowtangent as ft
from flowtangent.models.kernels import RBFKernel
from flowtangent.models.surrogates import GaussianProcessSurrogate
from flowtangent.models.manifolds import LinearPCA, Passthrough
from flowtangent.models.design import ActiveSpace, Parameter, Requirement

TEST_DIR = Path("/Users/jordan/dev/EDEN/flowtangent/exp/XFOIL").parent
DATA_DIR = Path("/Users/jordan/dev/data/")

In [24]:
@lru_cache(maxsize=2048)
def surf_lookup(name:str):
    af = getattr(ft.airfoils, name)
    y_af = jnp.concatenate([af.y_upper, af.y_lower])
    return y_af

def name2surf(row:dict):
    af_name = row["foil_name"].item()
    row["y_af"] = surf_lookup(af_name)
    return row

### Import Dataset

In [ ]:
from functools import lru_cache
import jax.numpy as jnp

XFD = ft.utils.data.XArrayDataset(DATA_DIR / "xfoil_4D.zarr").to_tabular(["run", "alpha"])
# XFD = XFD.transform_variable(name2surf)

In [15]:
256 * 42

10752

In [32]:
XFD.full_vars

['bend_data', 'conditions', 'foil_name', 'panel_data', 'polar_data']

### Setup Manifold

In [ ]:
airfold = LinearPCA(physical_dim=256, latent_dim=16) # Airfoil Encoding
confold = Passthrough(dim=5) # Mach, Re, Flap Angle, Flap Loc., Alpha
mfold = airfold + confold

### Setup Surrogate

In [ ]:
krnl = RBFKernel(input_dim=21)
surr = GaussianProcessSurrogate(42, input_dim=21, output_dim=1, kernel=krnl)

### Setup Parameters and Requirements

In [ ]:
y_af = Parameter(name="y_af")
Mach = Parameter(name="Mach")
Re = Parameter(name="Re")
flap_loc = Parameter(name="flap_x/c")
flap_angle = Parameter(name="flap_angle")
alpha = Paramter(name="alpha")

theta = (y_af, Mach, re, flap_loc, flap_angle, alpha)

CL_max = Requirement(name="CL", lower_bound=1.2)
alpha_range = Requirement(name="alpha", lower_bound=0.0, upper_bound=5.0)
y_star = (CL_max, alpha_range)

### Setup Design Space

In [ ]:
airspace = ActiveSpace(
    dataset=XFD,
    manifold=mfold,
    surrogate=surr,
    parameters=theta,
    requirements=y_star
)